# 1. Row-Level Security (RLS) in Databricks

### Row-level security allows you to restrict access to specific rows of a table based on the user’s identity or role. Users will only see data they are authorized to access.

**Key Concepts**
- Filter predicate: Defines which rows a user can see.
- Dynamic filtering: You can filter rows dynamically based on the current user.
- Integration: Works with Databricks SQL, Delta Tables, and Unity Catalog.

**Implementation Approaches**
- Using Unity Catalog and Table Access Policies (Recommended for Databricks SQL)
- You define row filters in the Unity Catalog.

**Notes / Best Practices for RLS**
- Always use Unity Catalog for row-level security in production.
- Avoid embedding filtering logic in application code; it’s harder to maintain.
- Test access with different users to ensure proper enforcement.
- Can be combined with Column-Level Security for sensitive fields.

In [0]:
%sql
-- Prerequisits
CREATE GROUP data_analysts;

ALTER GROUP data_analysts ADD USER `shubham@company.com`;

GRANT USE SCHEMA ON SCHEMA sales_schema TO `data_analysts`;

GRANT SELECT ON TABLE sales_schema.sales TO `data_analysts`;

In [0]:
%sql
CREATE ROW ACCESS POLICY region_rls
AS (region STRING) RETURNS BOOLEAN ->
  CASE
    WHEN is_member('north_sales') AND region = 'NORTH' THEN TRUE
    WHEN is_member('south_sales') AND region = 'SOUTH' THEN TRUE
    ELSE FALSE
  END;


-- Step 2: Apply Policy to Table
ALTER TABLE sales
ADD ROW ACCESS POLICY region_rls ON (region);

Explanation:

Only rows where owner_user = current_user() will be visible.

current_user() automatically identifies the logged-in user.

Policies are centrally managed via Unity Catalog.

### Programmatic Filtering (Legacy / Spark)

You can filter data in PySpark dynamically based on a mapping table of users to allowed rows.

In [0]:
user = "shubham@example.com"
allowed_regions = ["North", "East"]

df = spark.read.table("sales")
filtered_df = df.filter(df.region.isin(allowed_regions))
filtered_df.show()

# 2. Column-Level Security (CLS) in Databricks

**Column-level security allows you to restrict access to specific columns of a table, hiding sensitive data like PII, salaries, or credit card numbers.**

**Key Concepts**
- Restrict visibility of sensitive columns while allowing access to other columns.
- Enforced via Unity Catalog’s column masking or GRANT statements.
- Implementation Approaches
- Column Masking with Unity Catalog
- You can mask sensitive columns for certain users:

**Notes / Best Practices for CLS**
- Use masking policies instead of removing columns from tables to avoid breaking queries.
- Column-level security works best with Unity Catalog.
- Combine CLS with RLS for fine-grained security.
- Regularly review access logs to ensure compliance.

In [0]:
%sql
-- Only HR group will see salary
CREATE MASKING POLICY salary_mask
AS (salary DOUBLE) RETURNS DOUBLE ->
  CASE
    WHEN is_member('hr_team') THEN salary
    ELSE NULL
  END;

ALTER TABLE employees
ALTER COLUMN salary
SET MASKING POLICY salary_mask;

Explanation:
- Only admins see the real SSN.
- Other users see masked values (XXX-XX-XXXX).

**Column Grants:-** You can grant access to only certain columns:

**Explanation-** Analysts can see non-sensitive columns but not salary.

In [0]:
GRANT SELECT (name, department) ON TABLE employees TO `analyst@company.com`;
REVOKE SELECT ON COLUMN salary FROM `analyst@company.com`;